# Study 9: 4-D inference — (length_scale, mu_0, sigma, beta_speaker) via Laplace + VBMC

The full model fit for study 9: the GP hyperparameters `(length_scale, mu_0, sigma)` plus the
RSA speaker rationality `beta` are inferred jointly, `phi = [log_ls, mu_0, log_sigma, log_beta]`
(`inference.run_vbmc`; lognormal-ish prior on beta around the design value 3). The per-feature
Beta-mixture linking shapes are profiled out at every evaluation.

Structure:
1. 4-D VBMC, fitted (profiled) linking shapes (`results/study9-fit/fitted-feature-shapes.npz`)
2. Forward pass at the posterior median
3. GP coherence field per condition at the posterior median (as `plot-gp-coherence-field.ipynb`)

Results are cached — delete a file under `results/study9-fit/` to refit.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

import sys
sys.path.insert(0, ".")

import json
import numpy as np
import matplotlib.pyplot as plt

import inference as inf   # Laplace + VBMC machinery (sets jax x64; import before other jax users)

MAX_EVALS = 400
RESULTS_DIR = os.path.join('results', 'study9-fit')
os.makedirs(RESULTS_DIR, exist_ok=True)

geom = inf.load_geometry()
J = geom['J']
test_feature_names, test_trait = geom['test_feature_names'], geom['test_trait']
responses_cond, _ = inf.load_responses('../../data/study9_prereg.csv')
emp_cond = {c: np.array(responses_cond[c]).mean(0) for c in inf.CONDITIONS}
N_cond   = {c: int(responses_cond[c].shape[0]) for c in inf.CONDITIONS}
for c in inf.CONDITIONS:
    print(f"  {c:<14}: {N_cond[c]} participants  mean rating={np.array(responses_cond[c]).mean():.3f}")
SAMPLE_KEYS = ('ls_samples', 'mu0_samples', 'sigma_samples', 'beta_samples')

def load_or_fit(path, fit_fn):
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True)
        fit = {k: z[k] for k in SAMPLE_KEYS}
        fit.update(json.loads(str(z['meta'])))
        print(f"loaded cached posterior from {path}  ({fit['convergence_status']})")
    else:
        fit = fit_fn()
        np.savez_compressed(path, **{k: fit[k] for k in SAMPLE_KEYS},
                            meta=json.dumps({k: fit[k] for k in ('elbo', 'func_count',
                                                                 'convergence_status', 'runtime_s')}))
        print(f"saved posterior to {path}")
    return fit

# fixed default linking shapes (used by the forward pass below)
cond_data = inf.prepare_all_conditions(geom, responses_cond)


## 1. 4-D VBMC, fitted (profiled) linking shapes

Same treatment as section 7a of the main notebook; beta now also enters the ratings-free
coherence mode that the per-feature shape fit conditions on. Cached to
`results/study9-fit/fitted-feature-shapes.npz`.

In [ ]:
fit_f = load_or_fit(os.path.join(RESULTS_DIR, 'fitted-feature-shapes.npz'),
                    lambda: inf.run_vbmc(geom, responses_cond, max_evals=MAX_EVALS))

## 2. Forward pass at the posterior median (fitted shapes)

Fitted per-feature shapes at the 4-D posterior median, then predicted vs empirical prevalence
per condition (as section 7c of the main notebook).

In [ ]:
LS_B, MU_B, SIG_B, BETA_B = (float(np.median(fit_f['ls_samples'])), float(np.median(fit_f['mu0_samples'])),
                             float(np.median(fit_f['sigma_samples'])), float(np.median(fit_f['beta_samples'])))
print(f"Posterior median: ls={LS_B:.3f}, mu_0={MU_B:.3f}, sigma={SIG_B:.3f}, beta={BETA_B:.3f}")

_, shapes_f, cond_data_f = inf.total_log_lik_free_shapes(
    geom, responses_cond, LS_B, MU_B, SIG_B, beta_speaker=BETA_B, return_shapes=True)
mean_kl_f  = shapes_f[:, 0] / (shapes_f[:, 0] + shapes_f[:, 1])
mean_nkl_f = shapes_f[:, 2] / (shapes_f[:, 2] + shapes_f[:, 3])

np.savez_compressed(os.path.join(RESULTS_DIR, 'fitted-link-shapes.npz'),
                    shapes=shapes_f, feature_names=np.array(test_feature_names),
                    theta=np.array([LS_B, MU_B, SIG_B, BETA_B]))
import csv as _csv
SHAPES_CSV = os.path.join(RESULTS_DIR, 'fitted-link-shapes.csv')
with open(SHAPES_CSV, 'w', newline='') as f:
    w = _csv.writer(f)
    w.writerow(['feature', 'trait', 'a_kl', 'b_kl', 'a_nkl', 'b_nkl', 'mean_kl', 'mean_nkl'])
    for j, name in enumerate(test_feature_names):
        w.writerow([name, test_trait[j], *[f'{v:.4f}' for v in shapes_f[j]],
                    f'{mean_kl_f[j]:.4f}', f'{mean_nkl_f[j]:.4f}'])
print(f'saved fitted shapes to {SHAPES_CSV} (+ .npz with theta)')

traits = np.array(test_trait)
tcolor = {'diet': 'tab:green', 'personality': 'tab:purple', 'physical': 'tab:orange'}

pz1_mode_f, pred_cond_f = {}, {}
for c in inf.CONDITIONS:
    _, y_hat, ok = inf.laplace_log_Z(cond_data_f[c], LS_B, MU_B, SIG_B, beta_speaker=BETA_B)
    pz1_mode_f[c] = 1.0 / (1.0 + np.exp(-y_hat[:J]))
    pred_cond_f[c] = pz1_mode_f[c]*mean_kl_f + (1-pz1_mode_f[c])*mean_nkl_f
    if not ok:
        print(f"  warning: {c} Laplace mode not cleanly converged")

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), sharex=True, sharey=True)
for ax, c in zip(axes, inf.CONDITIONS):
    for tr in tcolor:
        m = traits == tr
        ax.scatter(emp_cond[c][m], pred_cond_f[c][m], s=70, color=tcolor[tr],
                   edgecolor=('k' if tr == c else 'none'), linewidth=1.5, label=tr)
    ax.plot([0, 1], [0, 1], '--', color='gray', alpha=0.5); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_title(f"{c}  (matched outlined)"); ax.set_xlabel('empirical mean')
axes[0].set_ylabel("predicted E[rating]"); axes[0].legend(fontsize=8, title='test trait')
r_all_f = np.corrcoef(np.concatenate([emp_cond[c] for c in inf.CONDITIONS]),
                      np.concatenate([pred_cond_f[c] for c in inf.CONDITIONS]))[0, 1]
fig.suptitle(f'4-D fit (fitted shapes): predicted vs empirical prevalence  (pooled r={r_all_f:.3f})')
plt.tight_layout(); plt.show()

## 3. GP coherence field per condition at the posterior median

Same visualization as `plot-gp-coherence-field.ipynb`: contours are the ratings-free
GP-conditional `P(kind-linked)` given each condition's trained region, at the 4-D
posterior-median theta (including the fitted beta, which enters the speaker term of the
coherence mode). Test points are colored by empirical mean rating on the same color scale.

In [ ]:
all_x = np.vstack([np.array(geom['x_test'])] + [np.array(geom['x_train_cond'][c]) for c in inf.CONDITIONS])
pad = 0.08
(x_lo, y_lo), (x_hi, y_hi) = all_x.min(0) - pad, all_x.max(0) + pad
gx, gy = np.meshgrid(np.linspace(x_lo, x_hi, 80), np.linspace(y_lo, y_hi, 80))
X_grid = np.column_stack([gx.ravel(), gy.ravel()])
x_test_np = np.array(geom['x_test'])

fig, axes = plt.subplots(2, 2, figsize=(13.5, 12))
for ax, c in zip(axes.ravel(), inf.CONDITIONS):
    field    = inf.gp_coherence_field(geom, c, LS_B, MU_B, SIG_B, X_grid,
                                      beta_speaker=BETA_B).reshape(gx.shape)
    pz1_test = inf.gp_coherence_field(geom, c, LS_B, MU_B, SIG_B, x_test_np, beta_speaker=BETA_B)
    cs = ax.contourf(gx, gy, field, levels=20, cmap='Blues')
    cb = fig.colorbar(cs, ax=ax); cb.set_label('GP P(kind-linked)')

    x_tr = np.array(geom['x_train_cond'][c])
    ax.scatter(x_tr[:, 0], x_tr[:, 1], s=60, color='#16305e', edgecolor='k',
               zorder=3, label='train (kind-linked)')
    for (xx, yy), nm in zip(x_tr, geom['train_names_cond'][c]):
        ax.annotate(nm[:12], (xx, yy), fontsize=4, xytext=(2, 3), textcoords='offset points')
    ax.scatter(x_test_np[:, 0], x_test_np[:, 1], s=80, c=emp_cond[c], cmap='Blues', norm=cs.norm,
               edgecolor='k', linewidth=1.2, zorder=3, label='test (mean response)')

    r = np.corrcoef(pz1_test, emp_cond[c])[0, 1]
    ax.set_title(f"{c.capitalize()}  (n={N_cond[c]})\nr(field @ test, mean rating) = {r:+.2f}")
    ax.set_xlabel('Embedding Dim 1'); ax.set_ylabel('Embedding Dim 2')
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle(f"Posterior over Coherence in Embedding Space -- study9-fit\n"
             f"shared fitted GP: ls={LS_B:.3f}, mu_0={MU_B:+.3f}, sigma={SIG_B:.3f}, "
             f"beta={BETA_B:.3f} (inferred)", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.965])
out_png = os.path.join(RESULTS_DIR, 'gp-coherence-field.png')
fig.savefig(out_png, dpi=150, bbox_inches='tight')
print('saved', out_png)
plt.show()